In [ ]:
!pip install transformers torch scikit-learn -q

import pandas as pd
import torch
import warnings
warnings.filterwarnings("ignore")

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils import resample
from google.colab import drive

def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(s):
        if s < num_warmup_steps:
            return s / max(1, num_warmup_steps)
        return max(0.0, (num_training_steps-s) / max(1, num_training_steps-num_warmup_steps))
    return LambdaLR(optimizer, lr_lambda)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print("All imported!")

Device: cuda
All imported!


In [ ]:
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Multimodal/multibully_with_prompts.csv")

SEED = 42
train_val, test_df = train_test_split(df, test_size=0.20,
                     stratify=df["bully_label"], random_state=SEED)
train_df, val_df   = train_test_split(train_val, test_size=0.125,
                     stratify=train_val["bully_label"], random_state=SEED)

bully    = train_df[train_df["bully_label"]==1]
notbully = train_df[train_df["bully_label"]==0]
train_balanced = pd.concat([
    notbully,
    resample(bully, replace=True, n_samples=len(notbully), random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Train (balanced): {len(train_balanced)}")
print(f"Val             : {len(val_df)}")
print(f"Test            : {len(test_df)}")

Mounted at /content/drive
Train (balanced): 6996
Val             : 580
Test            : 1159


In [ ]:
# ── Only change from Task 1 code — roberta-base instead of xlm-roberta-base
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)

bully_id  = tokenizer.convert_tokens_to_ids(tokenizer.tokenize("bully")[0])
normal_id = tokenizer.convert_tokens_to_ids(tokenizer.tokenize("normal")[0])

print(f"Model          : {MODEL_NAME}  (English only)")
print(f"Device         : {device}")
print(f"bully  token id: {bully_id}")
print(f"normal token id: {normal_id}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Model          : roberta-base  (English only)
Device         : cuda
bully  token id: 428
normal token id: 21113


In [ ]:
PROMPT_COL = "prompt_A"

class PromptDataset(Dataset):
    def __init__(self, df, col=PROMPT_COL):
        self.prompts = df[col].tolist()
        self.labels  = df["bully_label"].tolist()
    def __len__(self): return len(self.prompts)
    def __getitem__(self, idx):
        enc = tokenizer(self.prompts[idx], return_tensors="pt",
                        truncation=True, max_length=128, padding="max_length")
        return {"input_ids"     : enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "label"         : self.labels[idx]}

train_loader = DataLoader(PromptDataset(train_balanced), batch_size=8, shuffle=True)
val_loader   = DataLoader(PromptDataset(val_df),         batch_size=8, shuffle=False)
test_loader  = DataLoader(PromptDataset(test_df),        batch_size=8, shuffle=False)

print(f"Template     : {PROMPT_COL}")
print(f"Train batches: {len(train_loader)}")

Template     : prompt_A
Train batches: 875


In [ ]:
def evaluate(loader):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        with torch.no_grad():
            logits = model(input_ids=ids, attention_mask=attn).logits
        for i in range(ids.shape[0]):
            pos = (ids[i]==tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
            if len(pos)==0:
                preds.append(0)
            else:
                ml = logits[i, pos[0], :]
                preds.append(1 if ml[bully_id]>ml[normal_id] else 0)
        trues.extend(batch["label"].tolist())
    return f1_score(trues, preds, average="macro"), trues, preds

print("Evaluate function ready!")

Evaluate function ready!


In [ ]:
NUM_EPOCHS = 5
optimizer  = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
scheduler  = get_linear_schedule_with_warmup(
    optimizer, 100, (len(train_loader)//2)*NUM_EPOCHS
)

best_f1, best_state = 0, None
scaler = torch.amp.GradScaler("cuda") if device=="cuda" else None

print(f"Training: RoBERTa (English only) | {PROMPT_COL} | {NUM_EPOCHS} epochs")
print("-" * 55)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for i, batch in enumerate(train_loader):
        ids    = batch["input_ids"].to(device)
        attn   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        target = torch.where(labels==1,
                             torch.tensor(bully_id,  device=device),
                             torch.tensor(normal_id, device=device))

        lbl = torch.full(ids.shape, -100, device=device)
        lbl[(ids==tokenizer.mask_token_id)] = target.repeat_interleave(
            (ids==tokenizer.mask_token_id).sum(dim=1))

        if scaler:
            with torch.amp.autocast("cuda"):
                loss = model(input_ids=ids, attention_mask=attn,
                             labels=lbl).loss / 2
            scaler.scale(loss).backward()
        else:
            loss = model(input_ids=ids, attention_mask=attn,
                         labels=lbl).loss / 2
            loss.backward()

        total_loss += loss.item() * 2

        if (i+1) % 2 == 0:
            if scaler:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        del ids, attn, labels
        if device=="cuda": torch.cuda.empty_cache()

    val_f1, _, _ = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Loss: {total_loss/len(train_loader):.4f} | "
          f"Val F1: {val_f1*100:.2f}%")

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"  ★ Best saved!")

model.load_state_dict(best_state)
print(f"\nDone! Best Val F1: {best_f1*100:.2f}%")

Training: RoBERTa (English only) | prompt_A | 5 epochs
-------------------------------------------------------
Epoch 1/5 | Loss: 1.4835 | Val F1: 56.08%
  ★ Best saved!
Epoch 2/5 | Loss: 0.4130 | Val F1: 56.71%
  ★ Best saved!
Epoch 3/5 | Loss: 0.2137 | Val F1: 57.67%
  ★ Best saved!
Epoch 4/5 | Loss: 0.1265 | Val F1: 57.33%
Epoch 5/5 | Loss: 0.0784 | Val F1: 56.55%

Done! Best Val F1: 57.67%


In [ ]:
test_f1, y_true, y_pred = evaluate(test_loader)

print("=== ABLATION 3 — RoBERTa vs XLM-RoBERTa ===")
print(f"\nRoBERTa (English only)     : F1 = {test_f1*100:.2f}%")
print(f"XLM-RoBERTa (multilingual) : F1 = 61.74%")
print(f"Difference                 : {(61.74 - test_f1*100):+.2f}%")
print()
print(classification_report(y_true, y_pred,
      target_names=["Not-Bully", "Bully"]))

bully_caught = sum(t==1 and p==1 for t,p in zip(y_true, y_pred))
print(f"Bully caught : {bully_caught} / {sum(t==1 for t in y_true)}")

=== ABLATION 3 — RoBERTa vs XLM-RoBERTa ===

RoBERTa (English only)     : F1 = 60.27%
XLM-RoBERTa (multilingual) : F1 = 61.74%
Difference                 : +1.47%

              precision    recall  f1-score   support

   Not-Bully       0.89      0.91      0.90      1000
       Bully       0.33      0.28      0.31       159

    accuracy                           0.82      1159
   macro avg       0.61      0.60      0.60      1159
weighted avg       0.81      0.82      0.82      1159

Bully caught : 45 / 159


In [ ]:
import os

save_path = "/content/drive/MyDrive/Multimodal/roberta_ablation3/"
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

pred_path = "/content/drive/MyDrive/Multimodal/predictions_roberta_ablation3.csv"
test_df_save = test_df.copy()
test_df_save["pred"] = y_pred
test_df_save["true"] = y_true
test_df_save.to_csv(pred_path, index=False)

print(f"Saved!")
print(f"RoBERTa F1     : {test_f1*100:.2f}%")
print(f"XLM-RoBERTa F1 : 61.74%")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved!
RoBERTa F1     : 60.27%
XLM-RoBERTa F1 : 61.74%
